In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv("new_result.csv")
df = df.drop(columns=["qafacteval_available", "qafacteval_score"])
df

In [ ]:
def classify(id_value):
    if 1 <= id_value <= 50:
        return "Type 0"
    elif 51 <= id_value <= 100:
        return "Type 1"
    elif 101 <= id_value <= 150:
        return "Type 2"
    else:
        return "Out of range"

df["type"] = df["id"].apply(classify)

rouge_cols = ["rouge1_f1", "rouge2_f1", "rougeLsum_f1"]
bleu_cols = ["bleu1", "bleu2", "bleu3", "bleu4"]
bertscore_cols= ["bertscore_f1"]

avg_rouge_all = df.groupby("type")[rouge_cols].mean()
avg_rouge_overall = avg_rouge_all.mean(axis=1)

avg_bleu_all = df.groupby("type")[bleu_cols].mean()
avg_bleu_overall = avg_bleu_all.mean(axis=1)

avg_bertscore_f1 = df.groupby("type")[bertscore_cols].mean()

# Create summary dataframe
summary_df = pd.DataFrame({
    'ROUGE': avg_rouge_overall,
    'BLEU': avg_bleu_overall,
    'BERTScore F1': avg_bertscore_f1['bertscore_f1']
})

print("Summary Statistics by Type:")
print("=" * 50)
print(summary_df.round(4))
print("\n" + "=" * 50)
print(f"\nTotal samples: {len(df)}")
print(f"Samples per type:\n{df['type'].value_counts().sort_index()}")

summary_df

In [ ]:
# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROUGE Chart
avg_rouge_overall.sort_index().plot(kind="barh", ax=axes[0], color='#2E86AB')
axes[0].set_xlabel("Average ROUGE Score", fontsize=11)
axes[0].set_ylabel("Type", fontsize=11)
axes[0].set_title("Average ROUGE per Type\n(mean of ROUGE-1, ROUGE-2, ROUGE-Lsum)", fontsize=12, fontweight='bold')
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

# BLEU Chart
avg_bleu_overall.sort_index().plot(kind="barh", ax=axes[1], color='#A23B72')
axes[1].set_xlabel("Average BLEU Score", fontsize=11)
axes[1].set_ylabel("Type", fontsize=11)
axes[1].set_title("Average BLEU per Type\n(mean of BLEU-1, BLEU-2, BLEU-3, BLEU-4)", fontsize=12, fontweight='bold')
axes[1].grid(axis="x", linestyle="--", alpha=0.5)

# BERTScore Chart
avg_bertscore_f1['bertscore_f1'].sort_index().plot(kind="barh", ax=axes[2], color='#F18F01')
axes[2].set_xlabel("Average BERTScore F1", fontsize=11)
axes[2].set_ylabel("Type", fontsize=11)
axes[2].set_title("Average BERTScore F1 per Type", fontsize=12, fontweight='bold')
axes[2].grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Combined comparison chart - Normal bar chart with percentage scores
fig, ax = plt.subplots(figsize=(10, 6))

# Convert scores to percentages
rouge_pct = summary_df['ROUGE'] * 100
bleu_pct = summary_df['BLEU'] * 100
bertscore_pct = summary_df['BERTScore F1'] * 100

# Sort by index to ensure Type 0, Type 1, Type 2 order
rouge_pct = rouge_pct.sort_index()
bleu_pct = bleu_pct.sort_index()
bertscore_pct = bertscore_pct.sort_index()

x = np.arange(len(summary_df.index))
width = 0.25

# Create vertical bars
bars1 = ax.bar(x - width, rouge_pct, width, label='ROUGE', color='#2E86AB')
bars2 = ax.bar(x, bleu_pct, width, label='BLEU', color='#A23B72')
bars3 = ax.bar(x + width, bertscore_pct, width, label='BERTScore F1', color='#F18F01')

ax.set_xlabel('Question Type', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Comparison of All Metrics by Type', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(rouge_pct.index)  # Type 0, Type 1, Type 2
ax.legend(loc='upper right')
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_ylim(0, max(rouge_pct.max(), bleu_pct.max(), bertscore_pct.max()) * 1.15)

# Add value labels on bars (as percentages)
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Detailed breakdown: Individual ROUGE metrics
fig, ax = plt.subplots(figsize=(10, 6))
avg_rouge_all.sort_index().plot(kind="barh", ax=ax, width=0.8)
ax.set_xlabel("Average ROUGE Score", fontsize=11)
ax.set_ylabel("Type", fontsize=11)
ax.set_title("Detailed ROUGE Metrics by Type", fontsize=12, fontweight='bold')
ax.legend(['ROUGE-1', 'ROUGE-2', 'ROUGE-Lsum'], loc='lower right')
ax.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# Detailed breakdown: Individual BLEU metrics
fig, ax = plt.subplots(figsize=(10, 6))
avg_bleu_all.sort_index().plot(kind="barh", ax=ax, width=0.8)
ax.set_xlabel("Average BLEU Score", fontsize=11)
ax.set_ylabel("Type", fontsize=11)
ax.set_title("Detailed BLEU Metrics by Type", fontsize=12, fontweight='bold')
ax.legend(['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4'], loc='lower right')
ax.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
